
### DASH-01. Dashboard Aggregate Data Preparation

**Objective:** Generate a compact Parquet dataset containing precomputed descriptive statistics for the PyQt6 dashboard.

**Input:** Previously prepared PaySim Train (60%), Validation (20%), and Test (20%) datasets.

**Procedure:** Read the required transaction attributes in batches, aggregate transaction counts, fraud labels, transaction amounts, and distributions by transaction type, simulation step, and transaction amount range.

**Output:** `dashboard_aggregates.parquet` and `dashboard_metadata.json`.

**Methodological consideration:** This process generates descriptive statistics from the prepared PaySim dataset. It does not train, evaluate, or execute the fraud detection model. The reported fraud counts are derived from the dataset's `isFraud` labels, not model predictions.


In [2]:

# DASH-01 — Mount Drive and prepare dependencies

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Chạy một lần nếu thiếu thư viện Parquet.
%pip install -q pyarrow

ROOT = Path(
    "/content/drive/Shareddrives/"
    "M(anchine) L(earning)/Output"
)

SCRIPT = Path("/content/build_dashboard_aggregates.py")

assert SCRIPT.is_file(), (
    "Hãy upload build_dashboard_aggregates.py "
    "vào mục Files của Colab."
)

assert (ROOT / "Train_Validation_Test").is_dir()

print("PASS: Dashboard preparation environment ready.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PASS: Dashboard preparation environment ready.



### DASH-02. Aggregate Generation and Integrity Verification

**Objective:** Generate and validate the compact dashboard dataset.

**Procedure:** Execute the aggregation script, verify record counts and fraud totals against the prepared PaySim dataset, and compare the totals across aggregation levels.

**Expected validation checks:**
- Total transactions: 6,362,620.
- Fraud transactions: 8,213.
- Consistent totals across all aggregation levels.
- Successful Parquet write and reload verification.

**Output directory:** `Output/Dashboard/`.

**Important:** Aggregation levels are alternative summaries of the same dataset and must not be added together.


In [3]:

# DASH-02 — Run dashboard aggregate generation

import sys
import subprocess

result = subprocess.run(
    [
        sys.executable,
        str(SCRIPT),
        "--root",
        str(ROOT),
        "--batch-size",
        "100000"
    ],
    check=True,
    text=True
)

DASHBOARD_DIR = ROOT / "Dashboard"

AGGREGATE_PATH = (
    DASHBOARD_DIR / "dashboard_aggregates.parquet"
)

METADATA_PATH = (
    DASHBOARD_DIR / "dashboard_metadata.json"
)

assert AGGREGATE_PATH.is_file()
assert METADATA_PATH.is_file()

print("PASS: Dashboard aggregate generated successfully.")
print("Parquet:", AGGREGATE_PATH)
print("Metadata:", METADATA_PATH)


PASS: Dashboard aggregate generated successfully.
Parquet: /content/drive/Shareddrives/M(anchine) L(earning)/Output/Dashboard/dashboard_aggregates.parquet
Metadata: /content/drive/Shareddrives/M(anchine) L(earning)/Output/Dashboard/dashboard_metadata.json



### DASH-03. Dashboard Data Verification and Preview

**Objective:** Verify that the generated Parquet file can be independently loaded and that its aggregation schema is suitable for dashboard visualization.

**Procedure:** Inspect the overall KPI row, review the transaction-type summary, and check the saved file size and metadata.

**Output:** Verified aggregate schema and example dashboard data ready for PyQt6 integration.


In [4]:

# DASH-03 — Load and preview dashboard data

import json
import pandas as pd

dashboard_df = pd.read_parquet(AGGREGATE_PATH)

with open(METADATA_PATH, encoding="utf-8") as f:
    dashboard_meta = json.load(f)

print("Aggregate rows:", len(dashboard_df))
print("File size (KB):", round(
    AGGREGATE_PATH.stat().st_size / 1024, 2
))

print("\nAggregation levels:")
print(
    dashboard_df["aggregation_level"].value_counts()
)

print("\nOverall KPIs:")
display(
    dashboard_df.loc[
        dashboard_df["aggregation_level"] == "overall"
    ]
)

print("\nStatistics by transaction type:")
display(
    dashboard_df.loc[
        dashboard_df["aggregation_level"] == "by_type",
        [
            "transaction_type",
            "total_transactions",
            "fraud_transactions",
            "total_amount",
            "fraud_rate_pct"
        ]
    ].sort_values(
        "total_transactions",
        ascending=False
    )
)

# Kiểm tra tổng số giao dịch theo từng nhóm.
overall = dashboard_df.loc[
    dashboard_df["aggregation_level"] == "overall"
].iloc[0]

assert int(overall["total_transactions"]) == 6_362_620
assert int(overall["fraud_transactions"]) == 8_213

for level in dashboard_df["aggregation_level"].unique():
    subset = dashboard_df[
        dashboard_df["aggregation_level"] == level
    ]
    assert int(subset["total_transactions"].sum()) == (
        int(overall["total_transactions"])
    )
    assert int(subset["fraud_transactions"].sum()) == (
        int(overall["fraud_transactions"])
    )

print("\nPASS: Dashboard aggregate data verified.")


Aggregate rows: 3484
File size (KB): 131.34

Aggregation levels:
aggregation_level
by_type_step      2729
by_step            743
by_amount_band       6
by_type              5
overall              1
Name: count, dtype: int64

Overall KPIs:


,aggregation_level,step,transaction_type,amount_band,total_transactions,fraud_transactions,legitimate_transactions,total_amount,fraud_amount,legitimate_amount,avg_amount,fraud_rate_pct
3483,overall,<NA>,<NA>,<NA>,6362620,8213,6354407,1.144393e+12,1.205642e+10,1.132337e+12,179861.903549,0.129082



Statistics by transaction type:


,transaction_type,total_transactions,fraud_transactions,total_amount,fraud_rate_pct
750,CASH_OUT,2237500,4116,3.944130e+11,0.183955
752,PAYMENT,2151495,0,2.809337e+10,0.000000
749,CASH_IN,1399284,0,2.363674e+11,0.000000
753,TRANSFER,532909,4097,4.852920e+11,0.768799
751,DEBIT,41432,0,2.271992e+08,0.000000



PASS: Dashboard aggregate data verified.
